# birdset benchmark
https://huggingface.co/datasets/DBD-research-group/BirdSet

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from datasets import Audio, load_dataset

dataset = load_dataset("DBD-research-group/BirdSet", "HSN")

# slice example
dataset["train"] = dataset["train"].select(range(500))

# the dataset comes without an automatic Audio casting, this has to be enabled via huggingface
# this means that each time a sample is called, it is decoded (which may take a while if done for the complete dataset)
# in BirdSet, this is all done on-the-fly during training and testing (since the dataset size would be too big if mapping and saving it only once)
dataset = dataset.cast_column("audio", Audio(sampling_rate=32_000))


# extract the first five seconds of each sample in training (not utilizing event detection)
# a custom decoding with soundfile, stating start and end would be more efficient (see BirdSet Code)
def map_first_five(sample):
    max_length = 160_000  # 32_000hz*5sec
    sample["audio"]["array"] = sample["audio"]["array"][:max_length]
    return sample


# train is now available as an array that can be transformed into a spectrogram for example
train = dataset["train"].map(map_first_five, batch_size=1000, num_proc=2)

# the test_5s dataset is already divided into 5-second chunks where each sample can have zero, one or multiple bird vocalizations (ebird_code labels)
test = dataset["test_5s"]

Map (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

In [4]:
train

Dataset({
    features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
    num_rows: 500
})

In [11]:
test

Dataset({
    features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
    num_rows: 12000
})